# 23 — Hull-White Short-Rate Model

- **ZCB pricing** under Hull-White
- **Caplet / cap pricing** (analytic)
- **FD bond option** pricing
- **BSM-HW hybrid** (equity with stochastic rates)
- Generalized (multi-factor) Hull-White
- AD: model parameter sensitivities

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
QL = load_quantlib()

In [ ]:
from ql_jax.models.shortrate.hull_white import (
    hull_white_bond_price, hull_white_short_rate_mean, hull_white_caplet_price)
from ql_jax.engines.swaption.hull_white import hw_swaption_price, hw_bond_option_price
from ql_jax.engines.fd.hull_white import fd_hull_white_bond_option
from ql_jax.engines.analytic.bsm_hull_white import bsm_hull_white_price

A = 0.10          # mean reversion
SIGMA = 0.01      # short-rate vol
R0 = 0.05         # initial short rate
FLAT_RATE = 0.05

disc_fn = lambda t: jnp.exp(-FLAT_RATE * t)

---
## 1. Zero-Coupon Bond Pricing

In [ ]:
maturities = [1, 2, 3, 5, 7, 10, 15, 20, 30]
hw_prices = [float(hull_white_bond_price(R0, A, SIGMA, 0.0, float(T), disc_fn)) for T in maturities]
flat_prices = [float(disc_fn(float(T))) for T in maturities]

print(f"{'T':>5s} {'HW P(0,T)':>12s} {'Flat':>12s} {'Diff':>12s}")
print("-" * 45)
for T, hw, fl in zip(maturities, hw_prices, flat_prices):
    print(f"{T:>5d} {hw:>12.8f} {fl:>12.8f} {abs(hw-fl):>12.2e}")

---
## 2. HW Caplet Pricing

In [ ]:
K_cap = 0.05
caplets = []
for t_reset in [0.5, 1.0, 2.0, 3.0, 5.0]:
    cp = float(hull_white_caplet_price(R0, A, SIGMA, K_cap, t_reset, t_reset + 0.25, disc_fn))
    caplets.append((t_reset, cp))
    print(f"  Caplet T_reset={t_reset:.1f} : {cp:.6f}")

---
## 3. Bond Option: Analytic vs FD

In [ ]:
T_OPT, T_BOND, FACE = 3.0, 5.0, 100.0
K_BOND = 88.0  # strike

bo_analytic = float(hw_bond_option_price(T_OPT, T_BOND, K_BOND/FACE, A, SIGMA, disc_fn, is_call=True)) * FACE
bo_fd = float(fd_hull_white_bond_option(FACE, K_BOND, T_OPT, T_BOND, R0, A, SIGMA, disc_fn, option_type=1))

print(f"Bond option (analytic) : {bo_analytic:.6f}")
print(f"Bond option (FD)       : {bo_fd:.6f}")
print(f"Difference             : {abs(bo_analytic - bo_fd):.2e}")

---
## 4. BSM-HW Hybrid (Equity + Stochastic Rates)

In [ ]:
S, K, T = 100.0, 100.0, 2.0
sigma_s, q = 0.25, 0.02

for rho_sr in [-0.5, 0.0, 0.5]:
    p = float(bsm_hull_white_price(S, K, T, R0, q, sigma_s, A, SIGMA, rho_sr, disc_fn, option_type=1))
    print(f"  BSM-HW call (ρ_sr={rho_sr:+.1f}) : {p:.6f}")

---
## 5. AD: Parameter Sensitivities

In [ ]:
def hw_caplet_fn(a, sigma):
    return hull_white_caplet_price(R0, a, sigma, K_cap, 2.0, 2.25, disc_fn)

d_da = float(jax.grad(hw_caplet_fn, argnums=0)(A, SIGMA))
d_dsig = float(jax.grad(hw_caplet_fn, argnums=1)(A, SIGMA))

print(f"dCaplet/da    : {d_da:.6f}")
print(f"dCaplet/dσ    : {d_dsig:.6f}")

# Parameter surface
a_vals = jnp.linspace(0.01, 0.5, 30)
sig_vals = jnp.linspace(0.005, 0.03, 30)

prices = np.zeros((len(sig_vals), len(a_vals)))
for i, sv in enumerate(sig_vals):
    for j, av in enumerate(a_vals):
        prices[i, j] = float(hull_white_caplet_price(R0, float(av), float(sv), K_cap, 2.0, 2.25, disc_fn))

plt.figure(figsize=(8, 6))
plt.contourf(np.array(a_vals), np.array(sig_vals)*100, prices, levels=20, cmap='viridis')
plt.colorbar(label='Caplet Price')
plt.xlabel('Mean Reversion (a)')
plt.ylabel('Vol σ (%)')
plt.title('HW Caplet Price Surface')
plt.show()

---
## 6. Exercises

1. **Multi-factor HW**: Use `GeneralizedHullWhite` with 2 factors and compare the term structure shape.
2. **Calibration**: Fit $a$ and $\sigma$ to a set of cap prices using `jax.grad`-based minimization.
3. **Swaption**: Price a 5Y×5Y swaption using `hw_swaption_price` for different vol levels.

---
*End of Notebook 23*